## **Central Superstore Date Warehouse**

#### **Notebook Purpose :-**
- Creating Fact & Dimention Tables from superstore data and load them into SQL Server and then design a star schema diagram

#### **Notebook Objectives :-**
- Load data and libraries
- Data Overview
- Cleaning data and make it proper to sql format
- Creating Dimension tables
- Create the fact table
- Connecting with SQL Server and load data into it


#### **Data Warehouse Structure :-**

- Fact Table: *fact_order*
- Dimension Tables: *dim_date , dim_customer,  dim_product,  dim_ship_mode,  and  dim_location*
<br>
<br>

## **Importing The Libraries & Data**

In [ ]:
import pandas as pd
import numpy as np

data_path= r'C:\Users\uo\DEPI\Technical\SQL-miniproject2\dataset\Central_Superstore.xlsx'
data = pd.read_excel(data_path)
print('Data shape:' ,data.shape)
print('\nData types:' ,'\n',data.dtypes)


Data shape: (2323, 21)

Data types: 
 Row ID                    int64
Order ID                    str
Order Date       datetime64[us]
Ship Date        datetime64[us]
Ship Mode                   str
Customer ID                 str
Customer Name               str
Segment                     str
Country                     str
City                        str
State                       str
Postal Code               int64
Region                      str
Product ID                  str
Category                    str
Sub-Category                str
Product Name                str
Sales                   float64
Quantity                  int64
Discount                float64
Profit                  float64
dtype: object


## **Data Overview**

In [5]:
data.head(2)

,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,15,US-2012-118983,2014-11-22,2014-11-26,Standard Class,HP-14815,Harold Pawlan,Home Office,United States,Fort Worth,...,76106,Central,OFF-AP-10002311,Office Supplies,Appliances,Holmes Replacement Filter for HEPA Air Cleaner...,68.810,5,0.8,-123.858
1,16,US-2012-118983,2014-11-22,2014-11-26,Standard Class,HP-14815,Harold Pawlan,Home Office,United States,Fort Worth,...,76106,Central,OFF-BI-10000756,Office Supplies,Binders,Storex DuraTech Recycled Plastic Frosted Binders,2.544,3,0.8,-3.816


In [6]:
cust_map = data.groupby("Customer ID")[["Customer Name", "Segment"]].nunique()
print("Customer IDs with > 1 name:", (cust_map["Customer Name"] > 1).sum())
print("Customer IDs with > 1 segment:", (cust_map["Segment"] > 1).sum())

Customer IDs with > 1 name: 0
Customer IDs with > 1 segment: 0


In [7]:
# make a copy of the data
df =data.copy()

# make columns name proper to sql 
df.columns = (
             df.columns
                .str.strip()
                .str.replace(' ','_',regex=False,)
                .str.replace('/','_',regex=False)
                .str.replace('-','_',regex=False) 
                .str.lower()
)
print(df.columns)

Index(['row_id', 'order_id', 'order_date', 'ship_date', 'ship_mode',
       'customer_id', 'customer_name', 'segment', 'country', 'city', 'state',
       'postal_code', 'region', 'product_id', 'category', 'sub_category',
       'product_name', 'sales', 'quantity', 'discount', 'profit'],
      dtype='str')


### **Data Cleaning**

In [8]:
# remove the spaces in each column values
col = ['order_id','ship_mode','customer_id','customer_name', 'segment', 'country', 'city', 'state', 'region', 
       'product_id', 'category', 'sub_category','postal_code','product_name']
for s in col:
    df[s] =df[s].astype(str).str.strip()

In [9]:
df.isnull().sum()

row_id           0
order_id         0
order_date       0
ship_date        0
ship_mode        0
customer_id      0
customer_name    0
segment          0
country          0
city             0
state            0
postal_code      0
region           0
product_id       0
category         0
sub_category     0
product_name     0
sales            0
quantity         0
discount         0
profit           0
dtype: int64

## **Create Dimension Tables :-**
### *-date dim*

In [10]:
## concat the order_date and ship_date 
dates = pd.concat([df['order_date'],df['ship_date']]).drop_duplicates().sort_values()

print(dates.head())
print(type(dates))

1809   2013-01-03
188    2013-01-04
1752   2013-01-07
188    2013-01-08
152    2013-01-09
dtype: datetime64[us]
<class 'pandas.Series'>


In [11]:
dim_date= pd.DataFrame({'full_date': dates})

dim_date['date_key'] = dim_date["full_date"].dt.strftime("%Y%m%d").astype(int)
dim_date['year'] =dim_date['full_date'].dt.year
dim_date['quarter'] =dim_date['full_date'].dt.quarter
dim_date['month'] =dim_date['full_date'].dt.month
dim_date['month_name'] =dim_date['full_date'].dt.month_name()
dim_date['day'] =dim_date['full_date'].dt.day
dim_date['day_name'] =dim_date['full_date'].dt.day_name()
dim_date['week_of_year'] =dim_date['full_date'].dt.isocalendar().week.astype(int)
dim_date['is_weekend'] =dim_date['full_date'].dt.dayofweek >=5

dim_date= dim_date[['date_key','full_date','year','quarter','month','month_name','day','day_name','week_of_year','is_weekend']].reset_index(drop=True)

print('Date dimension shape : ',dim_date.shape)
print('\n',dim_date.head(3))


Date dimension shape :  (1058, 10)

    date_key  full_date  year  quarter  month month_name  day  day_name  \
0  20130103 2013-01-03  2013        1      1    January    3  Thursday   
1  20130104 2013-01-04  2013        1      1    January    4    Friday   
2  20130107 2013-01-07  2013        1      1    January    7    Monday   

   week_of_year  is_weekend  
0             1       False  
1             1       False  
2             2       False  


### *-product dim*

In [12]:
dim_product = df[['product_id','category','sub_category','product_name']].drop_duplicates().sort_values('product_id').reset_index(drop=True)

dim_product.insert(0, "product_key", dim_product.index + 1)

print('Product dimension shape : ',dim_product.shape)
print('\n',dim_product.head(3))

Product dimension shape :  (1326, 5)

    product_key       product_id   category sub_category  \
0            1  FUR-BO-10000112  Furniture    Bookcases   
1            2  FUR-BO-10000362  Furniture    Bookcases   
2            3  FUR-BO-10000468  Furniture    Bookcases   

                                       product_name  
0  Bush Birmingham Collection Bookcase, Dark Cherry  
1                Sauder Inglewood Library Bookcases  
2           O'Sullivan 2-Shelf Heavy-Duty Bookcases  


### *-customer dim*

In [13]:
dim_customer = df[['customer_id','customer_name','segment']].drop_duplicates().sort_values('customer_id').reset_index(drop=True)
dim_customer.insert(0,'customer_key', dim_customer.index + 1)

print('Customer dimension shape : ',dim_customer.shape)
print('\n',dim_customer.head(3))

Customer dimension shape :  (629, 4)

    customer_key customer_id customer_name   segment
0             1    AA-10315    Alex Avila  Consumer
1             2    AA-10375  Allen Armold  Consumer
2             3    AA-10480  Andrew Allen  Consumer


### *-ship mode dim*

In [14]:
dim_ship_mode= df[['ship_mode']].drop_duplicates().reset_index(drop=True)
dim_ship_mode.insert(0,'ship_mode_key',dim_ship_mode.index +1)

print('ship_mode_dimension shape : ',dim_ship_mode.shape)
print('\n',dim_ship_mode.head())

ship_mode_dimension shape :  (4, 2)

    ship_mode_key       ship_mode
0              1  Standard Class
1              2    Second Class
2              3     First Class
3              4        Same Day


### *-location dim*

In [15]:
dim_location =(df[['country','city','state', 'region','postal_code']]
               .drop_duplicates()
               .sort_values(['country','city','state', 'region','postal_code'])
               .reset_index(drop=True)
)
dim_location.insert(0,'location_key',dim_location.index +1)

print('location_dimension shape : ',dim_location.shape)
print('\n',dim_location.head())

location_dimension shape :  (195, 6)

    location_key        country       city         state   region postal_code
0             1  United States   Aberdeen  South Dakota  Central       57401
1             2  United States    Abilene         Texas  Central       79605
2             3  United States      Allen         Texas  Central       75002
3             4  United States   Amarillo         Texas  Central       79109
4             5  United States  Ann Arbor      Michigan  Central       48104


### *-fact order table*

In [16]:
fact_order=df.merge(dim_customer,on=['customer_id','customer_name','segment'],how='left')\
             .merge(dim_product,on =['product_id','category','sub_category','product_name'],how='left')\
             .merge(dim_ship_mode,on='ship_mode',how='left')\
             .merge(dim_location,on=['country','city','state', 'region','postal_code'],how='left')\
             .merge(dim_date[["date_key","full_date"]].rename(columns={"date_key":"order_date_key","full_date":"order_date"}),
                      on="order_date", how="left") \
             .merge(dim_date [["date_key","full_date"]].rename(columns={"date_key":"ship_date_key","full_date":"ship_date"}),
                      on="ship_date", how="left") 


fact_order.info()

<class 'pandas.DataFrame'>
RangeIndex: 2323 entries, 0 to 2322
Data columns (total 27 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   row_id          2323 non-null   int64         
 1   order_id        2323 non-null   str           
 2   order_date      2323 non-null   datetime64[us]
 3   ship_date       2323 non-null   datetime64[us]
 4   ship_mode       2323 non-null   str           
 5   customer_id     2323 non-null   str           
 6   customer_name   2323 non-null   str           
 7   segment         2323 non-null   str           
 8   country         2323 non-null   str           
 9   city            2323 non-null   str           
 10  state           2323 non-null   str           
 11  postal_code     2323 non-null   str           
 12  region          2323 non-null   str           
 13  product_id      2323 non-null   str           
 14  category        2323 non-null   str           
 15  sub_category   

In [17]:
fact_order=fact_order[['row_id', 'order_id','sales', 'quantity', 'discount', 'profit','customer_key', 'product_key', 'ship_mode_key','location_key'
                     ,'order_date_key', 'ship_date_key']].rename(columns={'row_id':'sales_key'}).sort_values('sales_key').reset_index(drop=True)

fact_order.info()

<class 'pandas.DataFrame'>
RangeIndex: 2323 entries, 0 to 2322
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   sales_key       2323 non-null   int64  
 1   order_id        2323 non-null   str    
 2   sales           2323 non-null   float64
 3   quantity        2323 non-null   int64  
 4   discount        2323 non-null   float64
 5   profit          2323 non-null   float64
 6   customer_key    2323 non-null   int64  
 7   product_key     2323 non-null   int64  
 8   ship_mode_key   2323 non-null   int64  
 9   location_key    2323 non-null   int64  
 10  order_date_key  2323 non-null   int64  
 11  ship_date_key   2323 non-null   int64  
dtypes: float64(3), int64(8), str(1)
memory usage: 217.9 KB


## Connect With SQL Server

In [18]:
from sqlalchemy import create_engine, text

SQL_SERVER = "localhost"
SQL_DATBASE = "SuperstoreDW"
SQL_DRIVER = "ODBC Driver 17 for SQL Server"
USE_TRUSTED_CONNECTION = True

try:
    conn_str = (
        f"mssql+pyodbc://@{SQL_SERVER}/{SQL_DATBASE}"
        f"?driver={SQL_DRIVER.replace(' ','+')}&trusted_connection=yes"
    )

    engine = create_engine(conn_str, fast_executemany=True)
    print("Conected to SQL Server")
except Exption as e:
    print(e)

with engine.connect() as conn:
    result = conn.execute(text("SELECT @@VERSION"))
    print(result.fetchone()[0])

Conected to SQL Server
Microsoft SQL Server 2025 (RTM) - 17.0.1000.7 (X64) 
	Oct 21 2025 12:05:57 
	Copyright (C) 2025 Microsoft Corporation
	Standard Developer Edition (64-bit) on Windows 10 Pro 10.0 <X64> (Build 19045: )



In [19]:
ddl_statements = [
	"DROP TABLE IF EXISTS dbo.fact_order;",
	"DROP TABLE IF EXISTS dbo.dim_ship_mode;",
	"DROP TABLE IF EXISTS dbo.dim_product;",
	"DROP TABLE IF EXISTS dbo.dim_location;",
	"DROP TABLE IF EXISTS dbo.dim_customer;",
	"DROP TABLE IF EXISTS dbo.dim_date;",

	"""
	CREATE TABLE dbo.dim_date (
		date_key INT PRIMARY KEY,
		full_date DATE NOT NULL,
		year SMALLINT NOT NULL,
		quarter TINYINT NOT NULL,
		month TINYINT NOT NULL,
		month_name VARCHAR(15) NOT NULL,
		day TINYINT NOT NULL,
		day_name VARCHAR(15) NOT NULL,
		week_of_year TINYINT NOT NULL,
		is_weekend BIT NOT NULL
	);
	""",

	"""
	CREATE TABLE dbo.dim_customer (
		customer_key INT PRIMARY KEY,
		customer_id VARCHAR(20) NOT NULL,
		customer_name VARCHAR(100) NOT NULL,
		segment VARCHAR(15) NOT NULL
	);
	""",

	"""
	CREATE TABLE dbo.dim_product (
		product_key INT PRIMARY KEY,
		product_id VARCHAR(20) NOT NULL,
		product_name VARCHAR(250) NOT NULL,
		category VARCHAR(20) NOT NULL,
		sub_category VARCHAR(20) NOT NULL
	);
	""",

	"""
	CREATE TABLE dbo.dim_location (
		location_key INT PRIMARY KEY,
		country VARCHAR(50) NOT NULL,
		region VARCHAR(50) NOT NULL,
		state VARCHAR(50) NOT NULL,
		city VARCHAR(50) NOT NULL,
		postal_code INT NOT NULL
	);
	""",

	"""
	CREATE TABLE dbo.dim_ship_mode (
		ship_mode_key INT PRIMARY KEY,
		ship_mode VARCHAR(30) NOT NULL
	);
	""",

	"""
	CREATE TABLE dbo.fact_order (
		sales_key INT PRIMARY KEY,
		order_id VARCHAR(20) NOT NULL,
		order_date_key INT NOT NULL,
		ship_date_key INT NOT NULL,
		ship_mode_key INT NOT NULL,
		customer_key INT NOT NULL,
		location_key INT NOT NULL,
		product_key INT NOT NULL,
		sales DECIMAL(12,4) NOT NULL,
		quantity INT NOT NULL,
		discount DECIMAL(5,4) NOT NULL,
		profit DECIMAL(12,4) NOT NULL,
		FOREIGN KEY (order_date_key) REFERENCES dbo.dim_date(date_key),
		FOREIGN KEY (ship_date_key) REFERENCES dbo.dim_date(date_key),
		FOREIGN KEY (ship_mode_key) REFERENCES dbo.dim_ship_mode(ship_mode_key),
		FOREIGN KEY (customer_key) REFERENCES dbo.dim_customer(customer_key),
		FOREIGN KEY (location_key) REFERENCES dbo.dim_location(location_key),
		FOREIGN KEY (product_key) REFERENCES dbo.dim_product(product_key)
	);
	"""
]

with engine.begin() as conn:
	for stmt in ddl_statements:
		conn.execute(text(stmt))

print('tables created successfully')

tables created successfully


## Load Tables To SQL

In [20]:
load_order = [
    ("dim_date", dim_date),
    ("dim_ship_mode", dim_ship_mode),
    ("dim_customer", dim_customer),
    ("dim_location", dim_location),
    ("dim_product", dim_product),
    ("fact_order", fact_order)
]

for table_name, df in load_order:
    df.to_sql(table_name, con=engine, schema="dbo", if_exists="append",
             index=False, chunksize=1000)
    print(f"Loaded {len(df)} rows --into--> dbo.{table_name}")

Loaded 1058 rows --into--> dbo.dim_date
Loaded 4 rows --into--> dbo.dim_ship_mode
Loaded 629 rows --into--> dbo.dim_customer
Loaded 195 rows --into--> dbo.dim_location
Loaded 1326 rows --into--> dbo.dim_product
Loaded 2323 rows --into--> dbo.fact_order


In [21]:
query = """
    SELECT 
        f.order_id
        ,d.full_date AS order_date
        ,c.customer_name
        ,p.product_name
        ,l.country
        ,l.city
        ,l.state
        ,f.sales
        ,f.quantity
        ,f.discount
        ,f.profit
    FROM dbo.fact_order f
    JOIN dbo.dim_date d
        ON f.order_date_key = d.date_key
    JOIN dbo.dim_ship_mode sm
        ON f.ship_mode_key = sm.ship_mode_key
    JOIN dbo.dim_customer c
        ON f.customer_key = c.customer_key
    JOIN dbo.dim_location l
        ON f.location_key = l.location_key
    JOIN dbo.dim_product p
        ON f.product_key = p.product_key
"""

new_df = pd.read_sql(query, engine)
new_df.head(3)

,order_id,order_date,customer_name,product_name,country,city,state,sales,quantity,discount,profit
0,US-2012-118983,2014-11-22,Harold Pawlan,Holmes Replacement Filter for HEPA Air Cleaner...,United States,Fort Worth,Texas,68.810,5,0.8,-123.8580
1,US-2012-118983,2014-11-22,Harold Pawlan,Storex DuraTech Recycled Plastic Frosted Binders,United States,Fort Worth,Texas,2.544,3,0.8,-3.8160
2,CA-2011-105893,2013-11-11,Pete Kriz,"Stur-D-Stor Shelving, Vertical 5-Shelf: 72""H x...",United States,Madison,Wisconsin,665.880,6,0.0,13.3176


In [22]:
new_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2323 entries, 0 to 2322
Data columns (total 11 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   order_id       2323 non-null   str    
 1   order_date     2323 non-null   object 
 2   customer_name  2323 non-null   str    
 3   product_name   2323 non-null   str    
 4   country        2323 non-null   str    
 5   city           2323 non-null   str    
 6   state          2323 non-null   str    
 7   sales          2323 non-null   float64
 8   quantity       2323 non-null   int64  
 9   discount       2323 non-null   float64
 10  profit         2323 non-null   float64
dtypes: float64(3), int64(1), object(1), str(6)
memory usage: 199.8+ KB
